In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

spark = SparkSession.builder \
    .appName("FlumenData-UploadFile") \
    .master("spark://spark-master:7077") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Master: {spark.conf.get('spark.master')}")

Spark version: 4.0.1
Master: spark://spark-master:7077


In [2]:
# ===============================
# Lakehouse Spark Configuration
# ===============================

# ---------- MinIO (S3) ----------
spark.conf.set("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
spark.conf.set("spark.hadoop.fs.s3a.access.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.secret.key", "minioadmin")

spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
spark.conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

# tối ưu ghi S3
spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version","2")
spark.conf.set("spark.hadoop.fs.s3a.committer.name","directory")
spark.conf.set("spark.hadoop.fs.s3a.fast.upload","true")


# ---------- Delta Lake ----------
spark.conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# ---------- Hive Metastore ----------
spark.conf.set("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")

# ---------- Warehouse ----------

In [3]:
facebook_df = spark.read.parquet("s3a://lakehouse/raw/facebook_crawl.parquet")
output_df = spark.read.parquet("s3a://lakehouse/raw/output.parquet")

print("facebook_crawl schema:")
facebook_df.printSchema()
print("output schema:")
output_df.printSchema()

facebook_crawl schema:
root
 |-- post_id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- post_date: string (nullable = true)
 |-- post_url: string (nullable = true)
 |-- post_text: string (nullable = true)
 |-- image_urls: string (nullable = true)
 |-- video_urls: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- comment_reactions: string (nullable = true)
 |-- is_reply: string (nullable = true)
 |-- parent_comment: string (nullable = true)

output schema:
root
 |-- id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: timestamp_ntz (nullable = true)
 |-- url: string (nullable = true)
 |-- images: string (nullable = true)
 |-- content: string (nullable = true)



In [4]:
facebook_df = facebook_df.cache()
output_df = output_df.cache()

facebook_df.count()
output_df.count()

13

In [5]:
from pyspark.sql.functions import col, lower, when, sum, try_to_timestamp
from pyspark.sql.types import StringType

def cast_columns(df):
    string_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, StringType)
    ]

    agg_exprs = []
    for c in string_cols:
        agg_exprs += [
            sum(((col(c).isNull()) | (col(c) == "")).cast("int")).alias(f"{c}_null"),
            sum(lower(col(c)).isin("true","false","1","0","yes","no").cast("int")).alias(f"{c}_bool"),
            sum(col(c).rlike("^-?[0-9]+$").cast("int")).alias(f"{c}_int"),
            sum(col(c).rlike("^-?[0-9]*\\.?[0-9]+$").cast("int")).alias(f"{c}_float"),
            sum(try_to_timestamp(col(c)).isNotNull().cast("int")).alias(f"{c}_ts"),
        ]

    stats_row = df.agg(*agg_exprs).collect()[0].asDict()
    total_rows = df.count()

    for c in string_cols:
        nulls = stats_row[f"{c}_null"]
        bool_valid = stats_row[f"{c}_bool"]
        int_valid = stats_row[f"{c}_int"]
        float_valid = stats_row[f"{c}_float"]
        ts_valid = stats_row[f"{c}_ts"]
        
        if bool_valid + nulls == total_rows:
            df = df.withColumn(
                c,
                when(lower(col(c)).isin("true","1","yes"), True)
                .when(lower(col(c)).isin("false","0","no"), False)
            )
        elif int_valid + nulls == total_rows:
            df = df.withColumn(c, col(c).cast("bigint"))
        elif float_valid + nulls == total_rows:
            df = df.withColumn(c, col(c).cast("double"))
        elif ts_valid + nulls == total_rows:
            df = df.withColumn(c, try_to_timestamp(col(c)))

    return df

In [6]:
facebook_df = cast_columns(facebook_df)
output_df = cast_columns(output_df)

In [7]:
print("facebook_crawl schema:")
facebook_df.printSchema()
print("output schema:")
output_df.printSchema()

facebook_crawl schema:
root
 |-- post_id: long (nullable = true)
 |-- source: string (nullable = true)
 |-- post_date: boolean (nullable = true)
 |-- post_url: boolean (nullable = true)
 |-- post_text: string (nullable = true)
 |-- image_urls: string (nullable = true)
 |-- video_urls: boolean (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- comment_text: string (nullable = true)
 |-- comment_reactions: long (nullable = true)
 |-- is_reply: boolean (nullable = true)
 |-- parent_comment: string (nullable = true)

output schema:
root
 |-- id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- date: timestamp_ntz (nullable = true)
 |-- url: string (nullable = true)
 |-- images: string (nullable = true)
 |-- content: string (nullable = true)



In [8]:
spark.sql("CREATE DATABASE IF NOT EXISTS social_media LOCATION 's3a://lakehouse/warehouse/social_media.db'")

facebook_df.write.format("delta").mode("overwrite").saveAsTable("social_media.facebook_data")
print("Wrote table: social_media.facebook_data")

output_df.write.format("delta").mode("overwrite").saveAsTable("social_media.output_data")
print("Wrote table: social_media.output_data")

https://maven-central.storage-download.googleapis.com/maven2/ added as a remote repository with the name: repo-1
https://maven-central.storage-download.googleapis.com/maven2/ added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/usr/local/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
org.apache.hadoop#hadoop-hdfs added as a dependency
org.datanucleus#datanucleus-api-jdo added as a dependency
org.datanucleus#datanucleus-rdbms added as a dependency
org.datanucleus#javax.jdo added as a dependency
org.springframework#spring-core added as a dependency
org.springframework#spring-jdbc added as a dependency
org.antlr#antlr4-runtime added as a dependency
org.apache.derby#derby added as a dependency
org.apache.hive#hive-metastore added as a dependency
org.apache.hive#hive-exec added as a dependency
org.apache.hive#hiv

Wrote table: social_media.facebook_data


Wrote table: social_media.output_data


In [ ]:
spark.stop()
print("✓ Spark session stopped")